# In this notebook, we will apply data generator to monitor test outcome on Grafana dashboard.

In [1]:
#!pip install .

In [2]:
#!pip install tensorflow

In [3]:
# !pip install seaborn six urllib3

In [4]:
# TensorFlow and tf.keras to load the data
import tensorflow as tf
from tensorflow import keras

# Commonly used modules
import numpy as np
import os
import sys

# Images, plots, display, and visualization
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import cv2
import IPython
from six.moves import urllib

print(tf.__version__)

2.20.0


# The data set is Boston Housing Prices classification data set https://www.kaggle.com/vikrishnan/boston-house-prices
## Step 1: Pre-process the data (normalization)

In [5]:
(train_features, train_labels), (test_features, test_labels) = keras.datasets.boston_housing.load_data()

# get per-feature statistics (mean, standard deviation) from the training set to normalize by
train_mean = np.mean(train_features, axis=0)
train_std = np.std(train_features, axis=0)
train_features = (train_features - train_mean) / train_std
print("Number of features per sample=", np.shape(train_features)[1])

Number of features per sample= 13


In [6]:
# Do the same for test features
test_mean = np.mean(test_features, axis=0)
test_std = np.std(test_features, axis=0)
test_features = (test_features - test_mean) / test_std

## Step 2: Define a simple Neural network model

In [7]:
def nn_model():
    model = keras.Sequential([
        keras.layers.Dense(100, activation=tf.nn.relu, input_shape=[len(train_features[0])]),
        keras.layers.Dense(1)
    ])

    model.compile(optimizer=tf.optimizers.Adam(), 
                  loss='mse',
                  metrics=['mae', 'mse'])
    return model

## Next, lets train the model

In [8]:
#Now, lets train the model
model = nn_model()

early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=50)
history = model.fit(train_features, train_labels, epochs=1000, verbose=1, validation_split = 0.1,
                    callbacks=[early_stop])

hist = pd.DataFrame(history.history)
hist['epoch'] = history.epoch


C:\Users\Ankit\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/1000
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 580.4399 - mae: 22.0541 - mse: 580.4399 - val_loss: 477.3588 - val_mae: 20.7473 - val_mse: 477.3588
Epoch 2/1000
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 555.3970 - mae: 21.5189 - mse: 555.3970 - val_loss: 454.3577 - val_mae: 20.1945 - val_mse: 454.3577
Epoch 3/1000
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 530.5427 - mae: 20.9634 - mse: 530.5427 - val_loss: 430.5979 - val_mae: 19.6043 - val_mse: 430.5979
Epoch 4/1000
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 503.9957 - mae: 20.3590 - mse: 503.9957 - val_loss: 405.3450 - val_mae: 18.9545 - val_mse: 405.3450
Epoch 5/1000
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 476.4783 - mae: 19.7045 - mse: 476.4783 - val_loss: 377.9048 - val_mae: 18.2391 - val_mse: 377.9048
Epoch 6/1000
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 445.4280 - mae: 18.9557 - mse: 445.4280 - val_loss: 349.1676 - val_mae: 17.4566 - val_mse: 349.1676
Epoch 7/1000
12/12 ━━━━━━━━━━━━━━━━━

In [9]:
# show RMSE measure to compare to Kaggle leaderboard on https://www.kaggle.com/c/boston-housing/leaderboard
rmse_final = np.sqrt(float(hist['val_mse'].tail(1)))
print()
print('Final Root Mean Square Error on validation set: {}'.format(round(rmse_final, 3)))


Final Root Mean Square Error on validation set: 2.278


C:\Users\Ankit\AppData\Local\Temp\ipykernel_35236\826533548.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  rmse_final = np.sqrt(float(hist['val_mse'].tail(1)))


In [10]:
print(hist)

           loss        mae         mse    val_loss    val_mae     val_mse  \
0    580.439880  22.054087  580.439880  477.358795  20.747253  477.358795   
1    555.396973  21.518936  555.396973  454.357697  20.194513  454.357697   
2    530.542725  20.963440  530.542725  430.597931  19.604269  430.597931   
3    503.995697  20.358976  503.995697  405.345001  18.954498  405.345001   
4    476.478302  19.704477  476.478302  377.904816  18.239096  377.904816   
..          ...        ...         ...         ...        ...         ...   
542    4.497419   1.467959    4.497419    5.333849   1.914924    5.333849   
543    4.584441   1.493988    4.584441    5.228328   1.879783    5.228328   
544    4.500166   1.474854    4.500166    5.496508   1.942971    5.496508   
545    4.480648   1.464041    4.480648    5.163410   1.887767    5.163410   
546    4.488789   1.490184    4.488789    5.187214   1.866746    5.187214   

     epoch  
0        0  
1        1  
2        2  
3        3  
4        4

In [11]:
mse, _, _ = model.evaluate(test_features, test_labels)
rmse = np.sqrt(mse)
print('Root Mean Square Error on test set: {}'.format(round(rmse, 3)))

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 14.0150 - mae: 2.5725 - mse: 14.0150
Root Mean Square Error on test set: 3.744


## Now, lets generate more test samples from test data and monitor them, as priduction data sets

In [12]:
print(np.shape(train_features), np.shape(train_labels))
print(np.shape(test_features), np.shape(test_labels))

(404, 13) (404,)
(102, 13) (102,)


In [13]:
num=np.shape(test_labels)[0]
import random

In [14]:
# !pip install daiquiri GPUtil prometheus_client PyDrive distutils-pytest PyYAML

In [15]:
#Lets import the ml_monotor library
import ml_monitor
import time
import numpy as np

In [16]:
my_monitor = ml_monitor.Monitor()
my_monitor.start()

2026-02-08 09:37:14,531 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...


In [ ]:
epochs=100000
for i in range(epochs):
    indx=random.randint(0,num-1)
    x_batch=np.expand_dims(test_features[indx,:],axis=0)
    y_batch=np.expand_dims(test_labels[indx],axis=0)
    mse, _, _ = model.evaluate(x_batch, y_batch, verbose=0)
    my_monitor.monitor("n_rmse", np.sqrt(mse))

2026-02-08 09:37:19,543 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:37:24,554 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:37:29,573 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:37:34,586 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:37:39,593 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:37:44,598 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:37:49,607 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:37:54,615 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:37:59,620 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:38:04,624 [35236] INFO     ml_monitor.logging: Starting metrics logging thread...
2026-02-08 09:38:09,633 [35236] INFO    